# Traffic Sensor Data Cleaning Pipeline

Below is the test cell to read all the Excel files

In [107]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "common":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "DDWEB_Downloads"
excel_files = sorted(DATA_DIR.glob("*.xlsx"))

print(f"Found {len(excel_files)} Excel files")
for f in excel_files:
    print(f.name)

Found 23 Excel files
DDweb_Auftrag_31032026_1706.xlsx
DDweb_Standort_01042026_1658.xlsx
DDweb_VI_Rohdaten_01042026_1726.xlsx
DDweb_VI_Rohdaten_01042026_1727.xlsx
DDweb_VI_Rohdaten_09042026_0811.xlsx
DDweb_VI_Rohdaten_09042026_0812.xlsx
DDweb_VI_Rohdaten_09042026_0813.xlsx
DDweb_VI_Rohdaten_09042026_1924.xlsx
DDweb_VI_Rohdaten_09042026_1925.xlsx
DDweb_VI_Rohdaten_09042026_1927.xlsx
DDweb_VI_Rohdaten_09042026_1928.xlsx
DDweb_VI_Rohdaten_09042026_1929.xlsx
DDweb_VI_Rohdaten_09042026_1930.xlsx
DDweb_VI_Rohdaten_09042026_1931.xlsx
DDweb_VI_Rohdaten_09042026_1932.xlsx
DDweb_VI_Rohdaten_09042026_1933.xlsx
DDweb_VI_Rohdaten_09042026_1934.xlsx
DDweb_VI_Rohdaten_09042026_1935.xlsx
DDweb_VI_Rohdaten_09042026_1936.xlsx
DDweb_VI_Rohdaten_16102023_1722.xlsx
DDweb_VI_Rohdaten_31032026_1633.xlsx
DDweb_VI_Rohdaten_31032026_1642.xlsx
DDweb_VI_Rohdaten_31032026_1656.xlsx


In [108]:
df_Auftrag = pd.read_excel(excel_files[0])
df_Auftrag.head()

/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Startdatum,Enddatum,Beschreibung,Geräte-ID,Gerätetyp,Standorttitel,Stadt,Inhaber,Erstellt
0,2022-01-28 13:00:00,2023-04-29 11:00:00,Fahrtrichtung Nord,6429,DD.plus,Albanstraße Nr. 23 DD 6429,Berlin Tempelhof-Schöneberg,NaN,2022-02-28 13:23:57.410
1,2022-02-25 14:00:00,2023-03-18 08:00:00,Fahrtrichtung-Süd,5949,DD.plus,Bahnstraße Nr. 10 DD 5949,Berlin Tempelhof-Schöneberg,NaN,2022-02-28 13:26:39.890
2,2021-04-27 11:10:00,2049-01-01 00:00:00,i.H.-HNr. 4,5951,DD.plus,Boelkestraße Nr. 58 Fr-Ri. Süd DD 5951,Berlin -Tempelhof-Schöneberg,NaN,2023-07-28 15:41:38.778
3,2021-05-04 11:10:00,2049-01-01 00:00:00,Fahrtrichtung Nord,5950,DD.plus,Boelkestraße Nr. 65 Fr. Nord DD 5950,Berlin Tempelhof-Schöneberg,NaN,2021-05-05 10:12:33.113
4,2026-03-13 13:00:00,2049-01-01 00:00:00,DD 5949 Halker Zeile,5949,DD.plus,DD 5949 Halker Zeile,Berlin Tempelhof-Schöneberg,NaN,2026-03-17 09:44:25.166


In [109]:
# ── QA accumulator ──────────────────────────────────────────────────────────
# We populate this dict throughout the notebook; it is printed as a report at the end.
qa_report = {}

print("Configuration loaded.")

Configuration loaded.


In [110]:
# Ensure date columns are proper datetimes, not strings
df_Auftrag["Startdatum"] = pd.to_datetime(df_Auftrag["Startdatum"], errors="coerce")
df_Auftrag["Enddatum"]   = pd.to_datetime(df_Auftrag["Enddatum"],   errors="coerce")

#the Startdatum column — the date each deployment began. That range makes perfect sense: the oldest sensor was first deployed in 2017,
#and the most recently started deployment kicked off in March 2026.
#Sentinel dates used in Auftrag to mean "deployment still active"
ACTIVE_SENTINELS = {
    pd.Timestamp("2049-01-01"),
    pd.Timestamp("2100-01-01"),
}

df_Auftrag["is_active"] = df_Auftrag["Enddatum"].apply(
    lambda d: any(abs((d - s).days) < 2 for s in ACTIVE_SENTINELS) if pd.notna(d) else False
)
df_Auftrag["Enddatum_clean"] = df_Auftrag.apply(
    lambda row: pd.NaT if row["is_active"] else row["Enddatum"], axis=1
)

print(f"Auftrag rows: {len(df_Auftrag)}")
print(f"Unique device IDs in Auftrag: {df_Auftrag['Geräte-ID'].nunique()}")

Auftrag rows: 44
Unique device IDs in Auftrag: 30


In [111]:
df_Standort = pd.read_excel(excel_files[1])
df_Standort.head()
print(f"Standort rows: {len(df_Standort)}")

Standort rows: 45


---
## 1. Drop Unimplemented Columns on Ingestion

Five columns are structurally zero across all files because the corresponding
sensor features were never activated. They are dropped immediately on load
to keep the working dataframe clean. The reasons are documented here:

| Column | Reason for dropping |
|---|---|
| `Schall (dB)` | Always 0 — sound measurement not implemented |
| `Abstand (cm)` | Always 0 — lateral distance not implemented |
| `Fahrspur` | Always 0 — lane indicator, not applicable on single-lane residential streets |
| `Geschwindigkeit (km/h)` | Always 0 — point speed not implemented; actual speed is in entry/exit columns |
| `Richtung` | Always 1 — one device per direction so this carries no information |

In [112]:
raw_files = sorted(DATA_DIR.glob("DDweb_VI_Rohdaten_*.xlsx"))
print(f"Found {len(raw_files)} raw data file(s):")
for f in raw_files:
    print(f"  {f.name}")

Found 21 raw data file(s):
  DDweb_VI_Rohdaten_01042026_1726.xlsx
  DDweb_VI_Rohdaten_01042026_1727.xlsx
  DDweb_VI_Rohdaten_09042026_0811.xlsx
  DDweb_VI_Rohdaten_09042026_0812.xlsx
  DDweb_VI_Rohdaten_09042026_0813.xlsx
  DDweb_VI_Rohdaten_09042026_1924.xlsx
  DDweb_VI_Rohdaten_09042026_1925.xlsx
  DDweb_VI_Rohdaten_09042026_1927.xlsx
  DDweb_VI_Rohdaten_09042026_1928.xlsx
  DDweb_VI_Rohdaten_09042026_1929.xlsx
  DDweb_VI_Rohdaten_09042026_1930.xlsx
  DDweb_VI_Rohdaten_09042026_1931.xlsx
  DDweb_VI_Rohdaten_09042026_1932.xlsx
  DDweb_VI_Rohdaten_09042026_1933.xlsx
  DDweb_VI_Rohdaten_09042026_1934.xlsx
  DDweb_VI_Rohdaten_09042026_1935.xlsx
  DDweb_VI_Rohdaten_09042026_1936.xlsx
  DDweb_VI_Rohdaten_16102023_1722.xlsx
  DDweb_VI_Rohdaten_31032026_1633.xlsx
  DDweb_VI_Rohdaten_31032026_1642.xlsx
  DDweb_VI_Rohdaten_31032026_1656.xlsx


In [113]:
COLS_TO_DROP = [
    "Schall (dB)",
    "Abstand (cm)",
    "Fahrspur",
    "Geschwindigkeit (km/h)",
    "Richtung",
]

RENAME_MAP = {
    "Geräte-ID":                          "device_id",
    "Datum":                              "datum_raw",
    "Eintrittsgeschwindigkeit (km/h)":    "speed_entry",
    "Austrittsgeschwindigkeit (km/h)":    "speed_exit",
    "Länge (dm)":                         "laenge_dm",
    "Klasse":                             "klasse",
    "Fahrzeugklassen-Bezeichnung":        "klasse_label",
}

frames = []
for fpath in raw_files:
    _df = pd.read_excel(fpath, dtype={"Geräte-ID": str})
    _df["source_file"] = fpath.name          # keep provenance
    _df = _df.drop(columns=COLS_TO_DROP, errors="ignore")
    _df = _df.rename(columns=RENAME_MAP)
    frames.append(_df)

df = pd.concat(frames, ignore_index=True)

print(f"Combined dataframe: {len(df):,} rows × {df.shape[1]} columns")
print(f"Columns kept: {list(df.columns)}")
df.head(3)

/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Use

Combined dataframe: 671,731 rows × 8 columns
Columns kept: ['device_id', 'datum_raw', 'speed_entry', 'speed_exit', 'laenge_dm', 'klasse', 'klasse_label', 'source_file']


,device_id,datum_raw,speed_entry,speed_exit,laenge_dm,klasse,klasse_label,source_file
0,5951,2026-03-25 00:19:54,62,70,42,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx
1,5951,2026-03-25 00:21:52,49,51,38,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx
2,5951,2026-03-25 00:51:22,37,43,38,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx


---
## 2. Timestamp Parsing with Multi-Format Detection

Two timestamp formats exist in the wild across export batches:
- **ISO format** (`YYYY-MM-DD HH:MM:SS`) — used in all recent exports; pandas reads this automatically
- **Legacy string format** (`DD/MM/YYYY HH:MM:SS`) — used in the 2022/2023 export; pandas will
  silently misparse this if you call `to_datetime` without specifying `dayfirst=True`

plan: detect the format per file and parse accordingly, then extract `datum` (date) and
`stunde` (hour 0–23) as separate columns for aggregation.

In [114]:
def parse_datum_column(series: pd.Series) -> pd.Series:
    """
    Parse a Datum column that may contain either:
    - datetime64 objects (already parsed by pandas on load from recent files)
    - strings in 'DD/MM/YYYY HH:MM:SS' format (legacy export format)
    Returns a datetime64 Series. Values that cannot be parsed become NaT.
    """
    if pd.api.types.is_datetime64_any_dtype(series):
        return series

    parsed = pd.to_datetime(series, dayfirst=True, errors="coerce") # It's a string column. Try the legacy DD/MM/YYYY format first (dayfirst=True)
    return parsed


# Parse each source file's dates separately
parsed_parts = []
for fpath in raw_files:
    mask = df["source_file"] == fpath.name
    parsed_parts.append(parse_datum_column(df.loc[mask, "datum_raw"]))

df["datum_parsed"] = pd.concat(parsed_parts).sort_index()

# For time components analysis
df["datum"]  = df["datum_parsed"].dt.date           # calendar date - for daily aggregation
df["stunde"] = df["datum_parsed"].dt.hour           # for hourly aggregation
df["wochentag"] = df["datum_parsed"].dt.day_name()  # for weekday patterns

# Flag any rows where parsing failed
df["flag_unparseable_timestamp"] = df["datum_parsed"].isna()
n_bad_ts = df["flag_unparseable_timestamp"].sum()
qa_report["unparseable_timestamp"] = int(n_bad_ts)


print(f"Timestamp parsing complete.")
print(f"  Rows with unparseable timestamp (flagged, not dropped): {n_bad_ts}")
print(f"  Date range: {df['datum_parsed'].min()} → {df['datum_parsed'].max()}")
df[["source_file", "datum_raw", "datum_parsed", "datum", "stunde", "wochentag"]].head(4)

Timestamp parsing complete.
  Rows with unparseable timestamp (flagged, not dropped): 0
  Date range: 2022-03-01 00:02:16 → 2026-04-08 23:31:24


,source_file,datum_raw,datum_parsed,datum,stunde,wochentag
0,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:19:54,2026-03-25 00:19:54,2026-03-25,0,Wednesday
1,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:21:52,2026-03-25 00:21:52,2026-03-25,0,Wednesday
2,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:51:22,2026-03-25 00:51:22,2026-03-25,0,Wednesday
3,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 01:25:16,2026-03-25 01:25:16,2026-03-25,1,Wednesday


---
## 3. Validate Geräte-ID Against Reference Table

Every `Geräte-ID` in the raw data should appear in the Auftrag deployment table.
A mismatch means either (a) the sensor was deployed without being registered, or
(b) the georeferencing table has not been updated after a device was moved or replaced.
Flagged rows cannot be georeferenced and should be held back from spatial analysis.

In [115]:
# Build the set of all device IDs that have ever appeared in the Auftrag table.
known_device_ids = set(df_Auftrag["Geräte-ID"].astype(str).unique())

df["unknown_device_flagged"] = ~df["device_id"].astype(str).isin(known_device_ids)

unknown_ids = df.loc[df["unknown_device_flagged"], "device_id"].unique()
n_unknown_rows = df["unknown_device_flagged"].sum()
qa_report["unknown_device_id"] = int(n_unknown_rows)

print(f"Known device IDs in Auftrag: {len(known_device_ids)}")
print(f"Unique device IDs in raw data: {df['device_id'].nunique()}")
print(f"Unregistered device IDs: {list(unknown_ids)}")
print(f"Rows flagged as unknown device (not dropped): {n_unknown_rows:,}")

Known device IDs in Auftrag: 30
Unique device IDs in raw data: 10
Unregistered device IDs: []
Rows flagged as unknown device (not dropped): 0


In [116]:
# Helpful summary: which devices appear in raw data, and whether they're known
device_summary = (
    df.groupby("device_id")
    .agg(row_count=("datum_parsed", "count"))
    .assign(in_auftrag=lambda x: x.index.isin(known_device_ids))
    .sort_values("row_count", ascending=False)
)
display(device_summary)

,row_count,in_auftrag
device_id,,
6424,242262,True
7207,224202,True
6426,54424,True
5950,41020,True
8468,38554,True
6429,36020,True
5949,27799,True
7871,4680,True
5951,2606,True


---
## 4. Speed Plausibility Check

Speed is recorded as entry speed and exit speed. Flagged rows are logged but won't be dropped for this step. 


In [117]:
# ── Speed plausibility thresholds (adjustable) ───────────────────────────────────────────
SPEED_MAX_MOTORISED  = 100   # km/h — flag any motorised vehicle above this
SPEED_MIN_MOTORISED  = 3     # km/h — flag any motorised vehicle below this (fyi. under 10km/h might be unreliable)
SPEED_MAX_BICYCLE    = 40    # km/h — flag any bicycle above this

# Klasse codes that are motorised (i.e. NOT bicycle or unknown).
MOTORISED_CLASSES = {2, 3, 5, 7, 8, 9, 10, 11, 64}
BICYCLE_CLASS     = 230

def speed_flag(speed_col, klasse_col):
    """
    It will return True for rows where the speed value is implausible given the vehicle class.
    Motorised vehicles: flag if speed < SPEED_MIN_MOTORISED or > SPEED_MAX_MOTORISED
    Bicycles:          flag if speed > SPEED_MAX_BICYCLE
    At the same time, zero exit speed is flagged — it represents an undefined read, not a stopped vehicle.
    """
    is_motorised = klasse_col.isin(MOTORISED_CLASSES)
    is_bicycle   = klasse_col == BICYCLE_CLASS

    flagged_motor   = is_motorised & (
        (speed_col > SPEED_MAX_MOTORISED) | (speed_col < SPEED_MIN_MOTORISED)
    )
    flagged_bicycle = is_bicycle & (speed_col > SPEED_MAX_BICYCLE)
    flag_zero_exit    = speed_col == 0

    return flagged_motor | flagged_bicycle | flag_zero_exit

In [118]:
df["speed_entry_flagged"] = speed_flag(df["speed_entry"], df["klasse"])
df["speed_exit_flagged"] = speed_flag(df["speed_exit"], df["klasse"])

# A row is flagged if either entry or exit speed is flagged
df["flag_speed"] = df["speed_entry_flagged"] | df["speed_exit_flagged"]

n_speed_entry = df["speed_entry_flagged"].sum()
n_speed_exit  = df["speed_exit_flagged"].sum()
n_speed_any   = df["flag_speed"].sum()
qa_report["implausible_entry_speed"] = int(n_speed_entry)
qa_report["implausible_exit_speed"]  = int(n_speed_exit)
qa_report["implausible_speed_any"]   = int(n_speed_any)

print(f"Rows with implausible entry speed: {n_speed_entry:,}")
print(f"Rows with implausible exit speed:  {n_speed_exit:,}")
print(f"Rows with implausible speed (either): {n_speed_any:,}")

Rows with implausible entry speed: 8
Rows with implausible exit speed:  38,568
Rows with implausible speed (either): 38,576


In [119]:
if n_speed_any > 0:
    print("\nSample flagged rows (investigate before the exclusion):")
    display(df[df["flag_speed"]][
        ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "source_file"]
        ].tail(50)
        )


Sample flagged rows (investigate before the exclusion):


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,source_file
352361,8468,2026-02-27 22:28:44,7,Pkw,16,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352362,8468,2026-02-27 22:29:21,7,Pkw,36,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352363,8468,2026-02-27 22:30:18,7,Pkw,31,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352364,8468,2026-02-27 22:31:06,7,Pkw,36,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352365,8468,2026-02-27 22:32:09,7,Pkw,19,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352366,8468,2026-02-27 22:32:27,7,Pkw,18,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352367,8468,2026-02-27 22:33:58,230,Fahrrad,18,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352368,8468,2026-02-27 22:35:39,7,Pkw,30,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352369,8468,2026-02-27 22:36:50,7,Pkw,17,0,DDweb_VI_Rohdaten_09042026_1933.xlsx
352370,8468,2026-02-27 22:39:53,7,Pkw,34,0,DDweb_VI_Rohdaten_09042026_1933.xlsx


---
## 5. Duplicate Detection

A true duplicate is a row where essential/meaningful field is identical — same device,
same timestamp, same class, same speed, same length -> This might indicate a data transmission error 
(the device sent the same record twice).
Flagged but not dropped — review before removing.

In [120]:
DEDUP_COLS = [
    "device_id",
    "datum_parsed",   # timestamp to the second
    "klasse",
    "speed_entry",
    "speed_exit",
    "laenge_dm",
]

# keep=False marks ALL copies of a duplicate as True, so the full set of duplicated rows is visible for investigation.
df["flag_duplicate"] = df.duplicated(subset=DEDUP_COLS, keep=False)
n_dup = df["flag_duplicate"].sum()
qa_report["duplicate_rows"] = int(n_dup)

print(f"Rows flagged as duplicates: {n_dup:,}")

if n_dup > 0:
    print("\nDuplicate groups (sorted by device and timestamp):")
    display(
        df[df["flag_duplicate"]]
        .sort_values(["device_id", "datum_parsed"])
        [["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "laenge_dm", "source_file"]]
        .head(60)
    )

Rows flagged as duplicates: 120

Duplicate groups (sorted by device and timestamp):


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,laenge_dm,source_file
581460,6426,2022-09-02 16:08:36,230,Fahrrad,13,12,16,DDweb_VI_Rohdaten_16102023_1722.xlsx
581461,6426,2022-09-02 16:08:36,230,Fahrrad,13,12,16,DDweb_VI_Rohdaten_16102023_1722.xlsx
581895,6426,2022-09-02 18:00:04,230,Fahrrad,20,21,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
581896,6426,2022-09-02 18:00:04,230,Fahrrad,20,21,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
582252,6426,2022-09-02 19:50:56,230,Fahrrad,15,14,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
582253,6426,2022-09-02 19:50:56,230,Fahrrad,15,14,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
583658,6426,2022-09-03 16:31:46,230,Fahrrad,14,13,14,DDweb_VI_Rohdaten_16102023_1722.xlsx
583659,6426,2022-09-03 16:31:46,230,Fahrrad,14,13,14,DDweb_VI_Rohdaten_16102023_1722.xlsx
584453,6426,2022-09-07 16:21:40,230,Fahrrad,15,13,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
584454,6426,2022-09-07 16:21:40,230,Fahrrad,15,13,15,DDweb_VI_Rohdaten_16102023_1722.xlsx


---
## 6. Entry/Exit Speed Ratio Check

As suggested by our Traffic Engineer, a **proportional ratio** is more physically meaningful:
the faster of the two records should not exceed `SPEED_RATIO_MAX` times the
slower one, regardless of the absolute speed level.

Rows where either speed is zero are skipped here — they are already flagged
by the absolute speed check. 

In [126]:
import numpy as np
SPEED_RATIO_MAX      = 2.5   # max ratio of faster / slower speed (entry vs exit)

def flag_speed_ratio(entry: pd.Series, exit_: pd.Series, ratio_max: float) -> pd.Series:
    """
    Proportional speed delta check.
    Rows where either speed is zero are NOT flagged here — they are already caught by the absolute speed plausibility check.
    """
    both_positive = (entry > 0) & (exit_ > 0)
    faster = pd.concat([entry, exit_], axis=1).max(axis=1)
    slower = pd.concat([entry, exit_], axis=1).min(axis=1).replace(0, np.nan)
    ratio  = faster / slower
    return both_positive & (ratio > ratio_max)


# Compute ratio column for the check
pos_double_e = (df["speed_entry"] > 0) & (df["speed_exit"] > 0)
faster   = df[["speed_entry", "speed_exit"]].max(axis=1)
slower   = df[["speed_entry", "speed_exit"]].min(axis=1)
slower_safe = slower.replace(0, np.nan)

df["speed_ratio"] = np.where(pos_double_e, faster / slower_safe, np.nan)

df["flag_speed_delta"] = flag_speed_ratio(df["speed_entry"], df["speed_exit"], SPEED_RATIO_MAX)

n_delta = df["flag_speed_delta"].sum()
qa_report["large_speed_ratio"] = int(n_delta)

print(f"Rows where faster/slower speed ratio > {SPEED_RATIO_MAX}: {n_delta:,}")
print(f"  (ratio_max = {SPEED_RATIO_MAX} means entry cannot be more than "
    f"{SPEED_RATIO_MAX}x exit, and vice versa)\n")

if n_delta > 0:
    flagged = df[df["flag_speed_delta"]].copy()

    print("Ratio distribution on flagged rows:")
    print(flagged["speed_ratio"].describe())

    print("\nBreakdown by vehicle class:")
    delta_by_class = (
        flagged
        .groupby(["klasse", "klasse_label"])
        .agg(
            flagged_rows=("speed_ratio",  "count"),
            max_ratio=("speed_ratio",     "max"),
            mean_ratio=("speed_ratio",    "mean"),
            max_entry=("speed_entry",      "max"),
            max_exit=("speed_exit",        "max"),
        )
    )
    display(delta_by_class)

Rows where faster/slower speed ratio > 2.5: 1,295
  (ratio_max = 2.5 means entry cannot be more than 2.5x exit, and vice versa)

Ratio distribution on flagged rows:
count     1295.0
unique     159.0
top          3.0
freq       110.0
Name: speed_ratio, dtype: float64

Breakdown by vehicle class:


,,flagged_rows,max_ratio,mean_ratio,max_entry,max_exit
klasse,klasse_label,,,,,
2,PkwA,28,4.25,3.137418,33,46
3,Lkw,154,5.888889,3.09183,60,59
5,Bus,1,2.785714,2.785714,39,14
6,nk Kfz,3,3.0,2.863095,23,8
7,Pkw,199,11.833333,3.07114,142,71
8,LkwA,5,8.0,4.141667,24,104
9,Sattel-Kfz,1,2.75,2.75,22,8
10,Krad,103,5.333333,3.151932,234,88
11,Lfw,176,7.461538,3.103376,41,97


In [128]:
print("\nSample flagged rows:")
display(
    df[df["flag_speed_delta"]][
    ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "speed_ratio"]
    ].sort_values("speed_ratio", ascending=False).head(15)
)


Sample flagged rows:


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,speed_ratio
71481,8481,2025-12-01 11:32:24,7,Pkw,142,12,11.833333
581287,6426,2022-09-02 15:26:34,7,Pkw,111,10,11.1
115205,6424,2022-05-16 07:30:22,8,LkwA,13,104,8.0
115204,6424,2022-05-16 07:30:22,11,Lfw,13,97,7.461538
71558,8481,2025-12-01 13:01:12,7,Pkw,139,22,6.318182
113804,6424,2022-05-05 09:21:36,11,Lfw,8,50,6.25
115136,6424,2022-05-15 09:58:14,3,Lkw,9,53,5.888889
115349,6424,2022-05-17 09:24:18,7,Pkw,8,45,5.625
484955,7207,2025-11-06 05:19:56,230,Fahrrad,39,7,5.571429
114973,6424,2022-05-13 21:55:42,3,Lkw,8,43,5.375


---
## 7. Unclassifiable Vehicles (Klasse 6 and Klasse 250)

Two classification codes indicate problematic records:

- **Klasse 6** (`nk Kfz` — nicht klassifizierbar): the sensor could not classify the vehicle.
  A high rate per sensor suggests an alignment or obstruction problem.
- **Klasse 250 (partially obscured)** (`Kfz`)

In [129]:
UNCLASSIFIABLE_CODES = {6, 250}   # row-level unclassifiable

df["flag_unclassifiable"]   = df["klasse"].isin(UNCLASSIFIABLE_CODES)
n_unclass  = df["flag_unclassifiable"].sum()
qa_report["unclassifiable_rows"]    = int(n_unclass)

print(f"Klasse 6 / 250 (unclassifiable, row-level): {n_unclass:,} rows")

Klasse 6 / 250 (unclassifiable, row-level): 86 rows


In [130]:
# Per-sensor breakdown for Klasse 6/250 — high rates indicate hardware problems
if n_unclass > 0:
    print("\nKlasse 6/250 per device (as % of that device's total rows):")
    device_total = df.groupby("device_id").size().rename("total")
    device_unclass = df[df["flag_unclassifiable"]].groupby("device_id").size().rename("unclassifiable")
    unclass_rate = pd.concat([device_total, device_unclass], axis=1).fillna(0)
    unclass_rate["pct"] = (unclass_rate["unclassifiable"] / unclass_rate["total"] * 100).round(2)
    display(unclass_rate[unclass_rate["unclassifiable"] > 0].sort_values("pct", ascending=False))



Klasse 6/250 per device (as % of that device's total rows):


,total,unclassifiable,pct
device_id,,,
7871,4680,2.0,0.04
6424,242262,41.0,0.02
6426,54424,9.0,0.02
8468,38554,9.0,0.02
7207,224202,22.0,0.01
5949,27799,1.0,0.00
5950,41020,2.0,0.00


---
## 8. Multi-Location Deployment Check

A single device can only be in one place at a time - meaning the Auftrag table
records two overlapping deployment windows for the same `Geräte-ID` at different
locations.

This check operates entirely on the **Auftrag reference table**, not on the raw
sensor readings. It is run after reference tables are loaded, so no raw data is
needed. A clean result here means every reading can be unambiguously assigned to
exactly one location for any given timestamp.

In [131]:
# Check for overlapping deployment windows per device ──────────────
# Strategy: for each device, compare every pair of its deployment windows.
# Using the cleaned Enddatum (where sentinels are replaced with pd.Timestamp.max) to have open-ended active deployments are included in the comparison correctly.
overlap_records = []

for gid, group in df_Auftrag.groupby("Geräte-ID"):
    rows = group.sort_values("Startdatum").reset_index(drop=True)
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            a_start = rows.loc[i, "Startdatum"]
            a_end   = rows.loc[i, "Enddatum_clean"]
            b_start = rows.loc[j, "Startdatum"]
            b_end   = rows.loc[j, "Enddatum_clean"]

            if pd.isna(a_start) or pd.isna(b_start):
                continue # Skip if either start date is NaT

            if a_start < b_end and b_start < a_end: # Overlap condition: intervals [a_start, a_end) and [b_start, b_end) intersect
                overlap_records.append({
                    "device_id":      gid,
                    "location_A":     rows.loc[i, "Standorttitel"],
                    "start_A":        a_start,
                    "end_A":          rows.loc[i, "Enddatum"],   # show the raw end date
                    "location_B":     rows.loc[j, "Standorttitel"],
                    "start_B":        b_start,
                    "end_B":          rows.loc[j, "Enddatum"],
                    "overlap_start":  max(a_start, b_start),
                    "overlap_end":    min(a_end,   b_end),
                })

n_overlaps = len(overlap_records)
qa_report["overlapping_deployment_windows"] = n_overlaps

if n_overlaps == 0:
    print("✓ No overlapping deployment windows found. Every device has clean, non-overlapping location history.")
else:
    print(f"{n_overlaps} overlapping deployment window pair detected:\n")
    overlap_df = pd.DataFrame(overlap_records)
    display(overlap_df)
    print("\nThese pairs indicate a data-entry error in the Auftrag table.")
    print("Readings that fall in the overlap period cannot be unambiguously")
    print("assigned to a single location. Investigate before spatial analysis.")

✓ No overlapping deployment windows found. Every device has clean, non-overlapping location history.


In [132]:
# A further step: flagging any raw sensor readings that fall inside an overlap period

def in_any_overlap(row, overlaps):
    gid = str(row["device_id"])
    ts  = row["datum_parsed"]
    if pd.isna(ts):
        return False
    return any(
        str(o["device_id"]) == gid and o["overlap_start"] <= ts <= o["overlap_end"]
        for o in overlaps
    )

df["flag_ambiguous_location"] = df.apply(in_any_overlap, overlaps=overlap_records, axis=1)
n_amb = df["flag_ambiguous_location"].sum()
qa_report["step9_raw_rows_in_overlap_window"] = int(n_amb)
print(f"\nRaw sensor rows falling inside an overlap period: {n_amb:,}")

# If no overlaps, add the column as all-False for consistency with the flag consolidation step
if "flag_ambiguous_location" not in df.columns:
    df["flag_ambiguous_location"] = False
    qa_report["raw_rows_in_overlap_window"] = 0


Raw sensor rows falling inside an overlap period: 0


---
## Geolocation Enrichment - Coordinates via Standorttitel

### After several tests, the Standorttitel is the right join key

There is a clean path: the `Standorttitel` column exists in **both** files and
I tested and confirmed to match exactly for all 44 Auftrag entries. This is the structured location
identifier the portal uses internally — it is what gets copied from Standort into Auftrag
when a new deployment is registered. You can join on it directly with no fuzzy matching.

### What we build

1. **`device_location_lookup`** — a lookup table: one row per deployment window, giving
   `device_id`, `start`, `end`, `lat`, `lon`, and clean address fields.
2. **`df` enriched** — the main sensor dataframe gets `lat`, `lon`, `strasse`, and
   `standorttitel` columns added by matching each reading's `(device_id, datum_parsed)`
   to the correct deployment window.

One extra flag is added: `flag_no_coords` — set when a reading's device/timestamp
combination produced no coordinate match (e.g. because it was already outside all
known deployment windows, or the Standort entry has missing coordinates).

In [133]:
# ── Build the device to location lookup table ───────────────────────
# Join Auftrag to Standort on Standorttitel (exact match).
# Keep only the columns needed for enrichment.

STANDORT_COLS = [
    "Standorttitel",
    "Benutzer Position Lat",
    "Benutzer Position Long",
    "Straße",
    "Hausnummer",
    "Postleitzahl",
    "Fahrtrichtung",
]

device_location_lookup = (
    df_Auftrag
    .merge(df_Standort[STANDORT_COLS], on="Standorttitel", how="left",)
    .rename(columns={
        "Geräte-ID":              "device_id",
        "Startdatum":             "deploy_start",
        "Enddatum_clean":         "deploy_end",
        "Standorttitel":          "standorttitel", #location title
        "Benutzer Position Lat":  "lat",
        "Benutzer Position Long": "lon",
        "Straße":                 "strasse",
        "Hausnummer":             "hausnummer",
        "Postleitzahl":           "postleitzahl", #post code
        "Fahrtrichtung":          "fahrtrichtung", #driving direction
    })
    [["device_id", "deploy_start", "deploy_end", "is_active",
        "standorttitel", "strasse", "hausnummer", "postleitzahl",
        "fahrtrichtung", "lat", "lon"]]
    .sort_values(["device_id", "deploy_start"])
    .reset_index(drop=True)
)

# Sanity check
no_coords = device_location_lookup["lat"].isna() | (device_location_lookup["lat"] == 0)
print(f"Lookup table rows: {len(device_location_lookup)}")
print(f"Rows with missing coordinates in Lookup Table: {no_coords.sum()}")
print("\nFull device → location lookup table:")
display(device_location_lookup)

Lookup table rows: 44
Rows with missing coordinates in Lookup Table: 0

Full device → location lookup table:


,device_id,deploy_start,deploy_end,is_active,standorttitel,strasse,hausnummer,postleitzahl,fahrtrichtung,lat,lon
0,34,2020-03-24 00:00:00,2021-05-10 00:00:00,False,Eisenacher Straße Nr. 106 DD 55,Eisenacher Straße,106,10781,Winterfeldtstraße,52.495950,13.349657
1,4848,2020-03-18 10:39:00,2022-12-03 10:00:00,False,Goltzstraße Nr. 45 G.-Nr.: 4848,Goltzstzraße,45,12307,Mellener Straße,52.385545,13.405569
2,5949,2022-02-25 14:00:00,2023-03-18 08:00:00,False,Bahnstraße Nr. 10 DD 5949,Bahnstraße,10,12277,Hranitzkystrae,52.422564,13.374872
3,5949,2023-03-18 11:10:00,2026-03-13 11:00:00,False,Wehnertstraße DD 5949,Wehnertstraße,45,NaN,Inzestraße,52.412517,13.373737
4,5949,2026-03-13 13:00:00,NaT,True,DD 5949 Halker Zeile,Halker Zeile,12,12305,Kettinger Straße,52.411605,13.393485
5,5950,2021-05-04 11:10:00,NaT,True,Boelkestraße Nr. 65 Fr. Nord DD 5950,Boelckestraße,65,12101,Nord -Loewenhardtdamm,52.478045,13.376915
6,5951,2021-04-27 11:10:00,NaT,True,Boelkestraße Nr. 58 Fr-Ri. Süd DD 5951,Boelckestraße,5,12101,Süd -Werner-Voß-Damm,52.478091,13.376559
7,5952,2021-11-13 14:00:00,NaT,True,Körtingstraße Nr. 45 G-Nr.: 5952,Körtingstraße,45,12107,Hirzeweg,52.433917,13.383631
8,5953,2021-11-13 15:00:00,NaT,True,Körtingstraße Nr. 42 DD 5953,Körtingstraße i.H. Ikarus-Grundschule,42,12107,Fritz-Werner-Straße -West-,52.433878,13.384897
9,6423,2022-02-25 13:00:00,2023-03-18 00:00:00,False,Friedenstraße DD 6423,Friedenstraße,23,12107,Fritz-Werner-Straße,52.438567,13.384607


### For each raw sensor reading we need to find which deployment window it falls in and attach the corresponding lat/lon and address.

Strategy: use a conditional merge.
   1. Left-join df to device_location_lookup on device_id.
   2. This produces one row per (reading × deployment window) pair for that device.
   3. Keep only the row where datum_parsed falls within [deploy_start, deploy_end].
  4. Readings with no matching window get NaN coordinates (flagged below).

### Step 1-2: cross-join on device ID only

In [134]:
df["device_id"] = df["device_id"].astype(str)
device_location_lookup["device_id"] = device_location_lookup["device_id"].astype(str)

# Step 1-2: cross-join on device ID only
df_geo = df.merge(
    device_location_lookup,
    on="device_id",
    how="left",
)

# Step 3: filter to the deployment window that contains this reading's timestamp
window_match = (
    df_geo["datum_parsed"].ge(df_geo["deploy_start"]) &
    df_geo["datum_parsed"].lt(df_geo["deploy_end"])
)

# Keep matched rows; also keep rows where no window matched at all
# (those will have NaN coords and get flagged — do not silently drop them)
matched     = df_geo[window_match]
unmatched   = df_geo[~df_geo.index.isin(matched.index)].drop_duplicates(subset=df.columns.tolist())

# For unmatched rows, keep only the original columns (no coord columns at all)
# so they survive the concat without duplicates
unmatched_base = unmatched[df.columns.tolist()].copy()
for col in ["lat", "lon", "standorttitel", "strasse", "hausnummer",
            "postleitzahl", "fahrtrichtung", "deploy_start", "deploy_end"]:
    unmatched_base[col] = pd.NA

---
## Consolidate Flag Columns & Build Flag Detail Table

We add a single `any_flag` boolean column so filtered analyses can easily
exclude all suspect rows in one expression. We also build a tidy `flag_detail`
table containing only the flagged rows, with a `flag_reasons` column listing
which checks triggered.

In [136]:
# All flag columns written during this notebook
FLAG_COLS = [
    "flag_unparseable_timestamp",
    "flag_unknown_device",
    "flag_unclassifiable",
    "flag_speed_entry",
    "flag_speed_exit",
    "flag_speed_any",
    "flag_duplicate",
    "flag_speed_delta",
]

# Keep only the flag columns that actually exist (defensive in case of future changes)
active_flags = [c for c in FLAG_COLS if c in df.columns]

df["any_flag"] = df[active_flags].any(axis=1)

# Build a human-readable 'flag_reasons' string per row
def summarise_flags(row):
    triggered = [c.replace("flag_", "") for c in active_flags if row.get(c)]
    return "; ".join(triggered) if triggered else ""

df["flag_reasons"] = df.apply(summarise_flags, axis=1)

# flag_detail: all rows with at least one flag, with key columns for investigation
flag_detail = df[df["any_flag"]][
    ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "speed_ratio", "laenge_dm", "source_file", "flag_reasons"]
    ].copy()

print(f"Total flagged rows (any flag): {df['any_flag'].sum():,} of {len(df):,} "
        f"({df['any_flag'].mean()*100:.1f}%)")
print(f"\nFlag detail table: {len(flag_detail):,} rows")
flag_detail.head(50)

Total flagged rows (any flag): 1,498 of 671,731 (0.2%)

Flag detail table: 1,498 rows


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,speed_ratio,laenge_dm,source_file,flag_reasons
488,5951,2026-03-26 07:50:40,3,Lkw,25,8,3.125,49,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
989,5951,2026-03-27 13:23:36,230,Fahrrad,27,8,3.375,13,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1212,5951,2026-03-27 22:15:52,230,Fahrrad,37,14,2.642857,11,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1302,5951,2026-03-28 12:33:42,230,Fahrrad,31,8,3.875,10,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1489,5951,2026-03-29 00:33:12,10,Krad,29,83,2.862069,24,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1672,5951,2026-03-29 16:01:54,10,Krad,28,85,3.035714,17,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1877,5951,2026-03-30 10:54:52,230,Fahrrad,34,8,4.25,17,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1921,5951,2026-03-30 13:24:32,230,Fahrrad,23,9,2.555556,18,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1938,5951,2026-03-30 14:09:58,3,Lkw,22,8,2.75,44,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
2323,5951,2026-03-31 12:28:40,230,Fahrrad,27,8,3.375,10,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta


---
## QA Report Summary

A single-glance summary of all checks run during this pipeline.

In [137]:
qa_report["total_rows_ingested"]  = len(df)
qa_report["total_rows_clean"]     = int((~df["any_flag"]).sum())
qa_report["source_files_loaded"]  = len(raw_files)

# Print as a tidy table
qa_df = pd.DataFrame.from_dict(qa_report, orient="index", columns=["count"])

STEP_LABELS = {
    "source_files_loaded":                    "Files loaded",
    "total_rows_ingested":                    "Total rows ingested",
    "total_rows_clean":                       "Rows with no flags",
    "total_rows_any_flag":                    "Rows with ≥1 flag",
    "unparseable_timestamp":                  "Unparseable timestamps",
    "unknown_device_id":                      "Rows with unregistered Geräte-ID",
    "unclassifiable_rows":                    "Rows Klasse 6/250 (unclassifiable)",
    "implausible_entry_speed":                "Implausible entry speed",
    "implausible_exit_speed":                 "Implausible exit speed",
    "implausible_speed_any":                  "Implausible speed (entry or exit)",
    "duplicate_rows":                         "Duplicate rows (all copies flagged)",
    "large_speed_delta":                      "Large entry/exit speed delta",
}

qa_df.index = [STEP_LABELS.get(k, k) for k in qa_df.index]
display(qa_df.style.format("{:,}"))

,count
Unparseable timestamps,0
Rows with unregistered Geräte-ID,0
Implausible entry speed,8
Implausible exit speed,"38,568"
Implausible speed (entry or exit),"38,576"
Duplicate rows (all copies flagged),120
large_speed_ratio,"1,295"
Rows Klasse 6/250 (unclassifiable),86
overlapping_deployment_windows,0
step9_raw_rows_in_overlap_window,0


---
## Clean Dataframe Ready for Analysis

`df_clean` excludes all flagged rows and is ready for aggregation.

In [138]:
df_clean = df[~df["any_flag"]].copy()

print(f"df       — full dataset with flags:  {len(df):,} rows")
print(f"df_clean — clean rows only:          {len(df_clean):,} rows")

df       — full dataset with flags:  671,731 rows
df_clean — clean rows only:          670,233 rows


---
# Part 2 — Hourly Aggregation

This section implements the **joining & aggregating process**: it
takes the cleaned, geo-enriched `df` produced by the cleaning pipeline above
and transform it to one row per **device × date × hour**.

### each row in the output table meaning: 
A single hour of traffic at a single sensor. It carries:
- **Class split counts** — one count column per vehicle class
- **Speed metrics** — mean entry speed, mean exit speed, mean bicycle speed
- **V85** — 85th-percentile entry speed excluding bicycles and motorcycles,
  the standard traffic-engineering design speed indicator
- **Hour-level quality flags** — promoted from the row-level flags in the
  cleaning pipeline. The hour is not dropped if flagged; the flags travel with it.

---
## Aggregation Configuration

In [139]:
# ── Klasse code → output column mapping ─────────────────────────────────────
# Each entry maps one output count column to the Klasse integer code it counts.
KLASSE_COUNT_COLS = {
    "count_pkw":     7,    # Pkw — Car
    "count_pkw_a":   2,    # PkwA — Car with trailer
    "count_lkw":     3,    # Lkw — Lorry
    "count_lkw_a":   8,    # LkwA — Lorry with trailer
    "count_sattel":  9,    # Sattel-Kfz — Articulated vehicle / HGV
    "count_bus":     5,    # Bus
    "count_krad":    10,   # Krad — Motorcycle
    "count_lfw":     11,   # Lfw — Delivery van
    "count_fahrrad": 230,  # Fahrrad — Bicycle
    "count_nk_kfz":  6,    # nk Kfz — Motor vehicle (unclassified)
    "count_kfz64":   64,   # Kfz — Motor vehicle (all)
}

# Motorised classes used for speed calculations.
# Klasse 64 excluded: speed value is valid but classification is not,
MOTORISED_SPEED_CLASSES = {2, 3, 5, 7, 8, 9, 10, 11}

# Classes excluded from V85 calculation: Bicycles and motorcycles have fundamentally different speed distributions
V85_EXCLUDED_CLASSES = {10, 230}   # Krad + Fahrrad

# V85 requires at least this many eligible speed readings to be considered reliable.
V85_MIN_SAMPLE = 5

# Columns used to group each hour
HOUR_GROUP_KEYS = ["device_id", "datum", "stunde", "wochentag"]

# Row-level flag columns from the cleaning pipeline that get promoted to hour level.
# A True at hour level means ≥1 row in that hour triggered that flag.
ROW_FLAG_COLS = [
    "flag_speed",
    "flag_unclassifiable",
    "flag_duplicate",
    "flag_speed_delta",
    "flag_unknown_device",
]

print("Aggregation configuration loaded.")
print(f"  Vehicle classes tracked: {len(KLASSE_COUNT_COLS)}")
print(f"  V85 excluded classes:    {V85_EXCLUDED_CLASSES}")
print(f"  V85 minimum sample:      {V85_MIN_SAMPLE}")

Aggregation configuration loaded.
  Vehicle classes tracked: 11
  V85 excluded classes:    {10, 230}
  V85 minimum sample:      5


---
## Pre-Aggregation Checks

In [140]:
# ── Assert required columns exist ────────────────────────────────────────────
REQUIRED_COLS = [
    "device_id", "datum_parsed", "datum", "stunde", "wochentag",
    "klasse", "speed_entry", "speed_exit", "any_flag",
]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, (
    f"Missing required columns: {missing}. "
    f"Ensure the cleaning pipeline above has run completely before this cell."
)
print("All required columns present.")

# ── Assert no null timestamps ──
n_null_ts = df["datum_parsed"].isna().sum()
if n_null_ts > 0:
    print(f"{n_null_ts:,} rows have null datum_parsed — these will be excluded "
        f"from aggregation because they cannot be assigned to an hour.")
    print(f" They were already flagged as flag_unparseable_timestamp in the cleaning step.")
else:
    print("No null timestamps.")

# ── Assert stunde is within 0–23 ──
bad_hours = df.loc[df["stunde"].notna(), "stunde"]
bad_hours = bad_hours[(bad_hours < 0) | (bad_hours > 23)]
assert len(bad_hours) == 0, f"stunde values outside 0–23 found: {bad_hours.unique()}"
print("All stunde values are valid (0–23).")

# ── Summarise what will be aggregated ──
df_agg_input = df.dropna(subset=["datum_parsed"]).copy()
print(f"\nDate range: {df_agg_input['datum'].min()} → {df_agg_input['datum'].max()}")
print(f"Devices:    {sorted(df_agg_input['device_id'].unique())}")

All required columns present.
No null timestamps.
All stunde values are valid (0–23).

Date range: 2022-03-01 → 2026-04-08
Devices:    ['5949', '5950', '5951', '6424', '6426', '6429', '7207', '7871', '8468', '8481']


---
## Hourly Aggregation Function

In [141]:
def aggregate_hour(grp: pd.DataFrame) -> pd.Series:
    """
    Aggregates one (device_id, datum, stunde) group into a single hourly row.

    ! All rows in the group are included — flagged rows are NOT pre-filtered.
    """
    result = {}
    result["count_total"] = len(grp)

    # Per-class counts
    for col_name, klass_code in KLASSE_COUNT_COLS.items():
        result[col_name] = int((grp["klasse"] == klass_code).sum())

    # Motorised total: excludes bicycles, Klasse 64, and unclassifiable
    result["count_motorised"] = int(grp["klasse"].isin(MOTORISED_SPEED_CLASSES).sum())

    # ── Speed metrics: exclude flagged rows within this group ─────────────────
    clean_subset   = ~grp["any_flag"] & ~grp["flag_speed_delta"]
    motor_subset  = grp["klasse"].isin(MOTORISED_SPEED_CLASSES) & clean_subset
    bike_subset    = (grp["klasse"] == 230) & clean_subset
    v85_subset    = (~grp["klasse"].isin(V85_EXCLUDED_CLASSES)) & clean_subset

    if motor_subset.sum() > 0:
        result["mean_speed_entry"] = float(grp.loc[motor_subset, "speed_entry"].mean())
        result["mean_speed_exit"]  = float(grp.loc[motor_subset, "speed_exit"].mean())
    else:
        result["mean_speed_entry"] = None
        result["mean_speed_exit"]  = None

    # Mean bicycle speed — separate metric, different population
    result["mean_speed_bicycle"] = (
        float(grp.loc[bike_subset, "speed_entry"].mean())
        if bike_subset.sum() > 0 else None
    )

    # V85 — 85th percentile entry speed - excludes Krad (10) and Fahrrad (230) per traffic engineering convention
    v85_speeds = grp.loc[v85_subset, "speed_entry"]
    n_v85      = len(v85_speeds)
    result["v85_entry"]       = round(float(v85_speeds.quantile(0.85)), 2) if n_v85 >= V85_MIN_SAMPLE else None
    result["n_v85_eligible"]  = int(n_v85)
    result["thin_v85_sample"] = bool(n_v85 < V85_MIN_SAMPLE)

    # ── Flags: promoted from row level to hour level ──────────────────────────
    # True at hour level means ≥1 row in this hour triggered that flag.
    # Only include flag columns that actually exist in the dataframe
    # (defensive against running on an older version of the pipeline output).
    existing_flags = [c for c in ROW_FLAG_COLS if c in grp.columns]

    result["flag_any"]              = bool(grp["any_flag"].any())
    result["flag_unclassifiable"]   = bool(grp["flag_unclassifiable"].any())       if "flag_unclassifiable"             in grp.columns else False
    result["flag_speed_issues"]     = bool(grp["flag_speed"].any())            if "flag_speed"                  in grp.columns else False
    result["flag_duplicate"]        = bool(grp["flag_duplicate"].any())            if "flag_duplicate"                  in grp.columns else False
    result["n_flagged_rows"]        = int(grp["any_flag"].sum())

    return pd.Series(result)


print("aggregate_hour() function defined.")

aggregate_hour() function defined.


---
## Run Aggregation

In [142]:
import time
_t0 = time.time()

hourly_dataset = (
    df_agg_input
    .groupby(HOUR_GROUP_KEYS, sort=True)
    .apply(aggregate_hour)
    .reset_index()
)

# Enforce clean dtypes — groupby/apply can produce object columns for int counts
INT_COLS = (
    ["stunde", "count_total", "count_motorised", "n_v85_eligible", "n_flagged_rows"]
    + list(KLASSE_COUNT_COLS.keys())
)
for col in INT_COLS:
    if col in hourly_dataset.columns:
        hourly_dataset[col] = pd.to_numeric(hourly_dataset[col], errors="coerce").astype("Int64")

FLOAT_COLS = ["mean_speed_entry", "mean_speed_exit", "mean_speed_bicycle", "v85_entry"]
for col in FLOAT_COLS:
    hourly_dataset[col] = pd.to_numeric(hourly_dataset[col], errors="coerce")

BOOL_COLS = [
    "thin_v85_sample", "flag_any", "flag_unclassifiable",
    "flag_speed_issues", "flag_duplicate",
    ]
for col in BOOL_COLS:
    if col in hourly_dataset.columns:
        hourly_dataset[col] = hourly_dataset[col].astype(bool)

_elapsed = time.time() - _t0
print(f"Aggregation complete in {_elapsed:.1f}s")
print(f"gold_hourly shape: {hourly_dataset.shape[0]:,} rows × {hourly_dataset.shape[1]} columns")
print(f"\nColumn list:")
print(list(hourly_dataset.columns))

Aggregation complete in 11.8s
gold_hourly shape: 8,979 rows × 28 columns

Column list:
['device_id', 'datum', 'stunde', 'wochentag', 'count_total', 'count_pkw', 'count_pkw_a', 'count_lkw', 'count_lkw_a', 'count_sattel', 'count_bus', 'count_krad', 'count_lfw', 'count_fahrrad', 'count_nk_kfz', 'count_kfz64', 'count_motorised', 'mean_speed_entry', 'mean_speed_exit', 'mean_speed_bicycle', 'v85_entry', 'n_v85_eligible', 'thin_v85_sample', 'flag_any', 'flag_unclassifiable', 'flag_speed_issues', 'flag_duplicate', 'n_flagged_rows']


---
## Post-Aggregation Integrity + Sanity Checks

These verify the aggregation produced internally consistent output.

In [143]:
checks_passed = 0

# ── 1. No duplicate (device, date, hour) keys ────────────────────────────────
dup_keys = hourly_dataset.duplicated(subset=["device_id", "datum", "stunde"], keep=False)
if dup_keys.any():
    raise AssertionError(
        f" {dup_keys.sum()} rows in gold_hourly share the same (device_id, datum, stunde) key "
    )
print("Check 1: No duplicate (device_id, datum, stunde) keys.")
checks_passed += 1

# ── 2. count_total matches sum of per-class counts ───────────────────────────
class_sum = hourly_dataset[list(KLASSE_COUNT_COLS.keys())].sum(axis=1)
unaccounted = hourly_dataset["count_total"] - class_sum
if (unaccounted < 0).any():
    print(f"Check 2: {(unaccounted < 0).sum()} hours where class column sum exceeds count_total ")
    print(hourly_dataset[unaccounted < 0][["device_id","datum","stunde","count_total"] + list(KLASSE_COUNT_COLS.keys())].head())
else:
    print("Check 2: Per-class counts are consistent with count_total.")
    checks_passed += 1

# ── 3. stunde values are 0–23 ────────────────────────────────────────────────
bad_stunde = hourly_dataset["stunde"].dropna()
bad_stunde = bad_stunde[(bad_stunde < 0) | (bad_stunde > 23)]
assert len(bad_stunde) == 0, f"stunde values outside 0–23 in gold_hourly: {bad_stunde.unique()}"
print("Check 3: All stunde values are 0–23.")
checks_passed += 1

# ── 4. V85 is null for all thin-sample hours ─────────────────────────────────
thin_with_v85 = hourly_dataset[hourly_dataset["thin_v85_sample"] & hourly_dataset["v85_entry"].notna()]
assert len(thin_with_v85) == 0, \
    f"{len(thin_with_v85)} hours marked thin_v85_sample=True but have a non-null v85_entry"
print("Check 4: V85 is null for all thin-sample hours.")
checks_passed += 1

# ── 5. Total row count sanity ────────────────────────────────────────────────
assert len(hourly_dataset) <= len(df_agg_input), \
    f"gold_hourly has more rows ({len(hourly_dataset)}) than input ({len(df_agg_input)}) "
print("Check 5: Row count is consistent.")
checks_passed += 1

print(f"All {checks_passed}/5 post-aggregation checks passed.")

Check 1: No duplicate (device_id, datum, stunde) keys.
Check 2: Per-class counts are consistent with count_total.
Check 3: All stunde values are 0–23.
Check 4: V85 is null for all thin-sample hours.
Check 5: Row count is consistent.
All 5/5 post-aggregation checks passed.


## Summary Report for Aggregation Step

In [144]:
print("HOURLY Aggregation SUMMARY")
print(f"  Total hourly rows:      {len(hourly_dataset):,}")
print(f"  Unique devices:         {hourly_dataset['device_id'].nunique()}")
print(f"  Date range:             {hourly_dataset['datum'].min()} → {hourly_dataset['datum'].max()}")
print(f"  Total passages counted: {hourly_dataset['count_total'].sum():,}")

print(f"\n--- Modal split (all hours combined) ---")
modal_totals = {}
for col, code in KLASSE_COUNT_COLS.items():
    total = int(hourly_dataset[col].sum())
    if total > 0:
        modal_totals[col] = total
grand_total = sum(modal_totals.values())
for col, total in sorted(modal_totals.items(), key=lambda x: -x[1]):
    pct = total / grand_total * 100 if grand_total > 0 else 0
    print(f"  {col:<20} {total:>8,}   ({pct:.1f}%)")

print(f"\n--- Speed summary (all hours, motorised) ---")
print(f"  Mean entry speed (motorised):  {hourly_dataset['mean_speed_entry'].mean():.1f} km/h")
print(f"  Mean V85 (motorised, excl. Krad/Fahrrad):  {hourly_dataset['v85_entry'].mean():.1f} km/h")

print(f"\n--- V85 reliability ---")
v85_valid = hourly_dataset['v85_entry'].notna().sum()
v85_thin  = hourly_dataset['thin_v85_sample'].sum()
print(f"  Hours with valid V85:       {v85_valid:,} of {len(hourly_dataset):,} ({v85_valid/len(hourly_dataset)*100:.1f}%)")
print(f"  Hours with thin sample:     {v85_thin:,}  (<{V85_MIN_SAMPLE} eligible readings)")

print(f"\n--- Quality flags at hour level ---")
for flag_col in ["flag_any", "flag_unclassifiable", "flag_speed_issues", "flag_duplicate"]:
    if flag_col in hourly_dataset.columns:
        n = int(hourly_dataset[flag_col].sum())
        pct = n / len(hourly_dataset) * 100
        print(f"  {flag_col:<30} {n:>5,} hours  ({pct:.1f}%)")

print(f"\n--- Per-device row counts ---")
display(
    hourly_dataset.groupby("device_id").agg(
        total_hours=("stunde", "count"),
        total_passages=("count_total", "sum"),
        date_range_start=("datum", "min"),
        date_range_end=("datum", "max"),
        mean_v85=("v85_entry", "mean"),
        hours_flagged=("flag_any", "sum"),
    ).round(1)
)

HOURLY Aggregation SUMMARY
  Total hourly rows:      8,979
  Unique devices:         10
  Date range:             2022-03-01 → 2026-04-08
  Total passages counted: 671,731

--- Modal split (all hours combined) ---
  count_pkw             354,984   (52.8%)
  count_fahrrad         225,501   (33.6%)
  count_kfz64            36,020   (5.4%)
  count_lfw              27,328   (4.1%)
  count_krad             14,093   (2.1%)
  count_lkw               8,323   (1.2%)
  count_bus               2,709   (0.4%)
  count_pkw_a             2,098   (0.3%)
  count_lkw_a               318   (0.0%)
  count_sattel              271   (0.0%)
  count_nk_kfz               86   (0.0%)

--- Speed summary (all hours, motorised) ---
  Mean entry speed (motorised):  24.9 km/h
  Mean V85 (motorised, excl. Krad/Fahrrad):  30.4 km/h

--- V85 reliability ---
  Hours with valid V85:       7,194 of 8,979 (80.1%)
  Hours with thin sample:     1,785  (<5 eligible readings)

--- Quality flags at hour level ---
  flag_any    

,total_hours,total_passages,date_range_start,date_range_end,mean_v85,hours_flagged
device_id,,,,,,
5949,602,27799,2026-03-13,2026-04-07,36.8,41
5950,167,41020,2026-03-25,2026-03-31,35.4,36
5951,163,2606,2026-03-25,2026-03-31,39.3,10
6424,4173,242262,2022-03-01,2023-03-30,26.0,634
6426,567,54424,2022-09-02,2022-09-29,24.6,110
6429,661,36020,2023-04-01,2023-04-28,37.0,21
7207,1392,224202,2025-09-01,2025-11-29,36.4,163
7871,448,4680,2026-03-14,2026-04-08,23.3,19
8468,801,38554,2025-12-01,2026-02-27,32.7,9


In [145]:
# Full table preview — all columns
display(hourly_dataset.head(25))

,device_id,datum,stunde,wochentag,count_total,count_pkw,count_pkw_a,count_lkw,count_lkw_a,count_sattel,...,mean_speed_exit,mean_speed_bicycle,v85_entry,n_v85_eligible,thin_v85_sample,flag_any,flag_unclassifiable,flag_speed_issues,flag_duplicate,n_flagged_rows
0,5949,2026-03-13,13,Friday,105,92,2,0,0,0,...,30.950495,17.500000,37.00,100,False,True,False,False,False,2
1,5949,2026-03-13,14,Friday,156,128,0,0,0,0,...,30.573333,21.166667,36.00,147,False,False,False,False,False,0
2,5949,2026-03-13,15,Friday,137,112,0,0,0,0,...,31.447761,18.666667,36.00,128,False,False,False,False,False,0
3,5949,2026-03-13,16,Friday,97,77,1,0,0,0,...,30.842105,17.000000,36.50,91,False,False,False,False,False,0
4,5949,2026-03-13,17,Friday,93,84,0,0,0,0,...,31.184783,22.000000,36.00,91,False,False,False,False,False,0
5,5949,2026-03-13,18,Friday,90,78,0,0,0,0,...,30.916667,19.000000,35.00,83,False,False,False,False,False,0
6,5949,2026-03-13,19,Friday,66,60,0,0,0,0,...,33.047619,22.000000,36.00,62,False,False,False,False,False,0
7,5949,2026-03-13,20,Friday,53,46,0,0,0,0,...,33.560000,17.666667,39.80,49,False,False,False,False,False,0
8,5949,2026-03-13,21,Friday,38,34,0,0,0,0,...,31.228571,17.333333,36.90,35,False,False,False,False,False,0
9,5949,2026-03-13,22,Friday,29,23,0,0,0,0,...,33.615385,21.666667,41.75,24,False,False,False,False,False,0
